# Multi-LoRA email workshop: data and routing

**Primary question:** Can a text-only router select finance, travel, and general operations specialists without consulting the email's true labels?

This notebook follows the CLM notebook pattern: evaluation contract, secure configuration, fail-fast data checks, a small baseline, charts, and reproducibility artifacts.

**Time:** 45-60 minutes. **Hardware:** CPU. Run cells top to bottom. Data lineage and routing are model-independent; the model-profile choice intentionally starts in notebook 02. See the [workshop guide](../README.md) for setup and the complete agenda.

## Evaluation contract

- The source is **multi-label**, despite its repository name: preserve every label, including `Business` + `Reminders`.
- Pin the dataset revision; validate subject/body/labels before splitting.
- Collapse normalized exact duplicates with identical labels; quarantine conflicting duplicates and record their IDs.
- Make an iterative multi-label 70/15/15 split. Fit vocabulary and classifiers on **train only**; tune thresholds on **validation only**.
- Never route using ground-truth labels. Keep the test split sealed until the final comparison.
- This CPU baseline is a real TF-IDF classifier, **not** LoRA training, an LLM, or a proxy for GPU quality.

In [ ]:
from __future__ import annotations

import os
import platform
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import email_workshop
from email_workshop.data import DOMAINS, LABELS, label_matrix, load_splits, prepare, write_json
from email_workshop.evaluation import lexical_predictions, score
from email_workshop.routing import fit_router

WORKSHOP_ROOT = Path(email_workshop.__file__).resolve().parents[1]
ARTIFACT_DIR = WORKSHOP_ROOT / Path(os.getenv('EMAIL_WORKSHOP_RUN', 'artifacts/run-2026'))
RANDOM_STATE = 2026
TARGET_ROUTING_RECALL = 0.95
RUN_PREPARATION = True
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.spines.top': False, 'axes.spines.right': False})
print(f'Python: {platform.python_version()} | Kernel: {sys.executable}')
print(f'Artifacts: {ARTIFACT_DIR}')

## Secure configuration

The dataset and small base model are public; no token is required. Hugging Face can use `HF_TOKEN` if your environment requires it. Never print the token. This notebook downloads public data but does not upload emails or models.

Artifacts and `.env` files are ignored by Git. The raw data lives in the local Hugging Face cache; split files contain email text and should stay local. The publisher labels the dataset MIT, but that alone does not establish that every email is synthetic or PII-free.

In [ ]:
if (ARTIFACT_DIR / 'data' / 'manifest.json').exists():
    splits, manifest = load_splits(ARTIFACT_DIR)
    print('Reusing verified immutable splits.')
elif RUN_PREPARATION:
    manifest = prepare(ARTIFACT_DIR, seed=RANDOM_STATE)
    splits, manifest = load_splits(ARTIFACT_DIR)
else:
    raise FileNotFoundError('Run preparation once or select an existing EMAIL_WORKSHOP_RUN.')

display(pd.Series({key: manifest[key] for key in ['raw_rows', 'unique_rows', 'rows_removed', 'dataset_revision']}))
display(pd.DataFrame({name: item['label_counts'] for name, item in manifest['splits'].items()}).reindex(LABELS))
print('Quarantine audit:', manifest['deduplication'])

## Inspect imbalance and overlapping labels

At the pinned revision, the source has 2,105 rows. One text appears twice with conflicting labels, so the preparation policy retains 2,103 usable emails. With seed 2026, the expected counts are 1,473 train, 312 validation, and 318 test. Iterative stratification prioritizes label proportions, so sizes need not equal exact rounded percentages.

Only training emails are used below. Subject/body contents are deliberately not displayed.

In [ ]:
train_targets = label_matrix(splits['train'])
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
pd.Series(train_targets.sum(axis=0), index=LABELS).sort_values().plot.barh(ax=axes[0], title='Training label counts')
pd.Series(train_targets.sum(axis=1)).value_counts().sort_index().plot.bar(ax=axes[1], title='Labels per training email')
axes[1].set_xlabel('Number of applicable labels')
plt.tight_layout()
plt.show()
cooccurrence = pd.DataFrame(train_targets.T @ train_targets, index=LABELS, columns=LABELS)
display(cooccurrence)

## Define the adapter contracts

- **Generalist:** all ten labels; the fair one-adapter baseline.
- **Operations:** the eight non-finance/non-travel labels.
- **Finance:** `Finance & Bills` or `[]`.
- **Travel:** `Travel & Bookings` or `[]`.

Finance and travel are separate adapters, so an invoice and a flight confirmation can route differently. A travel invoice may activate both. This is a domain fan-out problem, not forced top-1 classification.

Each adapter trains on the **same full training partition**. A specialist receives the intersection of the email's labels and its allowed labels, including `[]` for off-domain emails. Positive-only specialist training would hide routing errors and teach constant-positive answers.

In [ ]:
domain_table = []
for name, labels in DOMAINS.items():
    positives = sum(bool(set(row['labels']) & set(labels)) for row in splits['train'])
    domain_table.append({'adapter': name, 'allowed_labels': ', '.join(labels), 'positive_rows': positives, 'negative_rows': len(splits['train']) - positives})
display(pd.DataFrame(domain_table))

## Fit the text-only router and tune on validation

Fit TF-IDF unigrams/bigrams and one-vs-rest logistic regression on train. Choose the label threshold by validation micro-F1. A domain's routing score is the maximum probability among its labels; it is **not a calibrated union probability**.

Select the validation routing threshold with the fewest calls while retaining at least 95% domain recall. If no domain passes, use the highest-scoring domain. Do not change the threshold after looking at test results.

In [ ]:
router, router_metadata = fit_router(ARTIFACT_DIR, target_recall=TARGET_ROUTING_RECALL)
display(pd.Series({key: value for key, value in router_metadata.items() if key != 'trials'}))
trials = pd.DataFrame(router_metadata['trials'])
fig, ax = plt.subplots()
ax.plot(trials['mean_calls'], trials['domain_recall'], marker='o')
ax.axhline(TARGET_ROUTING_RECALL, color='tab:red', linestyle='--', label='Validation target')
ax.set(xlabel='Mean specialist calls per email', ylabel='Domain recall', title='Validation routing trade-off')
ax.legend()
plt.show()

## Measure the lexical baseline on validation

Report exact label-set match, micro/macro F1, Hamming loss, and per-label precision/recall. Multiclass log loss and a ten-way confusion matrix are not appropriate for overlapping labels.

Validation metrics are tuning diagnostics, not final held-out quality. The final notebook evaluates all systems on identical test IDs.

In [ ]:
validation_predictions = lexical_predictions(splits['validation'], router)
summary, per_label = score(validation_predictions)
display(pd.Series(summary))
display(pd.DataFrame(per_label).set_index('label'))
write_json(ARTIFACT_DIR / 'router' / 'validation_metrics.json', {'summary': summary, 'per_label': per_label})

## Synthetic routing smoke test

These are hand-written demonstration emails, not evaluation examples. No training labels are passed to `router.domains`. A wrong route is an observation to diagnose on validation, not something to overwrite with the known answer.

In [ ]:
examples = [
    ('Invoice due', 'Please pay the attached supplier invoice by Friday.'),
    ('Flight confirmation', 'Your flight and hotel reservations are confirmed.'),
    ('Travel receipt', 'Your airline booking is confirmed; the invoice is attached.'),
    ('Team meeting', 'Reminder: please attend tomorrow morning\'s project review.'),
]
display(pd.DataFrame([{'example': subject, 'selected_adapters': router.domains(subject, body)} for subject, body in examples]))

## Reproducibility and handoff

The data manifest includes the source checksum, pinned revision, taxonomy, conflict audit, seed, split hashes, and label counts. The router is refitted deterministically from train/validation instead of loading an unsafe pickle.

**Exit check:** Explain why 2,105 became 2,103; why exact-match accuracy is harder than per-label accuracy; why a travel invoice may need two specialists; and why all-negative finance predictions can appear accurate despite being useless.

Continue to [02: train adapters](02_train_email_adapters.ipynb).

In [ ]:
ledger = pd.DataFrame([
    {'artifact': 'data/manifest.json', 'purpose': 'Dataset lineage and immutable splits'},
    {'artifact': 'router/metadata.json', 'purpose': 'Validation-selected thresholds'},
    {'artifact': 'router/validation_metrics.json', 'purpose': 'Lexical validation baseline'},
])
ledger.to_csv(ARTIFACT_DIR / 'artifact_ledger.csv', index=False)
display(ledger)